# Sentiment Analysis - Mobile Phone Reviews
## Production-Ready NLP Pipeline

In [ ]:
# ===== 1. IMPORT LIBRARIES =====
import pandas as pd
import numpy as np
import re
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from textblob import TextBlob

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer,LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Download NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab',quiet=True)
nltk.download('stopwords' ,quiet=True)
nltk.download('wordnet' ,quiet=True)

# !pip install imbalanced-learn
# !pip install xgboost
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# ===== 2. LOAD DATA =====
df = pd.read_excel("dataset.xlsx")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# ===== 3. COMBINE TITLE AND BODY =====
df['text'] = df['title'] + " " + df['body']
print(df[['text', 'rating']].head())

In [ ]:
# ===== 4. DEFINE PREPROCESSING FUNCTION (REUSABLE) =====

# Initialize lemmatizer and stopwords ONCE (outside the function for efficiency)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Clean and preprocess a single text string.
    This function is used BOTH during training and prediction.
    """
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+", "", text)
    
    # Remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z]", " ", text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    
    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(tokens)

# Test the function
sample_text = "This product is AMAZING!!! I love it 😊 http://example.com"
print(f"Original: {sample_text}")
print(f"Cleaned: {preprocess_text(sample_text)}")

In [ ]:
# ===== 5. APPLY PREPROCESSING TO ALL DATA =====
df['clean_text'] = df['text'].apply(preprocess_text)
df[['text', 'clean_text']].head()

In [ ]:
# ===== 6. CREATE SENTIMENT LABELS =====

# Calculate polarity scores
df['polarity'] = df['clean_text'].apply(lambda x: TextBlob(x).sentiment.polarity)

# Define label function
def get_sentiment_label(polarity_score):
    if polarity_score > 0.2:
        return "Positive"
    elif polarity_score < -0.2:
        return "Negative"
    else:
        return "Neutral"

df['sentiment'] = df['polarity'].apply(get_sentiment_label)

# Check class distribution
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

# Visualize
df['sentiment'].value_counts().plot(kind='bar', color=['red', 'gray', 'green'])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
# ===== 7. PREPARE FEATURES AND LABELS =====
X = df['clean_text']
y = df['sentiment']

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")

In [ ]:
# ==========================================
# LABEL ENCODING
# ==========================================

label_encoder = LabelEncoder()

df['sentiment_encoded'] = label_encoder.fit_transform(df['sentiment'])

print("\nLabel Mapping:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name} --> {i}")

# Features and target
X = df['cleaned_review']
y = df['sentiment_encoded']

In [ ]:
# ===== 8. TRAIN-TEST SPLIT =====
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"\nTraining set distribution:\n{y_train.value_counts()}")

In [ ]:
# ===== 9. BUILD ML PIPELINE WITH SMOTE =====

# ===== TF-IDF VECTORIZATION =====

tfidf = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)

# Transform text data
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TF-IDF transformation complete!")
print("Training shape:", X_train_tfidf.shape)

# ==========================================
# CONVERT SPARSE MATRIX TO DENSE MATRIX
# ==========================================

X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

print("Converted sparse matrix to dense matrix!")

# ==========================================
# APPLY SMOTE
# ==========================================

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_dense,
    y_train
)

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())

# ==========================================
# XGBOOST MODEL
# ==========================================

model = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    
    subsample=0.8,
    colsample_bytree=0.8,
    
    random_state=42,
    eval_metric='mlogloss'
)

print("\nXGBoost model created successfully!")

In [ ]:
# ===== TRAIN MODEL =====

print("Training XGBoost model...")

model.fit(X_train_smote, y_train_smote)

print("✓ Training complete!")

In [ ]:
# ===== 11. EVALUATE MODEL =====

# Predictions
# ===== PREDICTIONS =====

y_pred = model.predict(X_test_dense)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n{'='*50}")
print(f"TEST ACCURACY: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"{'='*50}\n")

# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=['Negative', 'Neutral', 'Positive'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Neutral', 'Positive'],
            yticklabels=['Negative', 'Neutral', 'Positive'])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - Accuracy: {accuracy:.2%}")
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model,
    X_train_smote,
    y_train_smote,
    cv=5,
    scoring='accuracy'
)

print("\nCross Validation Scores:")
print(cv_scores)

print(f"\nMean CV Accuracy: {cv_scores.mean():.4f}")

In [ ]:
# ===== 13. TEST ON RAW INPUT (SIMULATING WEB APP) =====

def predict_sentiment(raw_text):

    # Preprocess
    cleaned = preprocess_text(raw_text)

    # TF-IDF transform
    vectorized = tfidf.transform([cleaned])

    # Dense conversion
    vectorized_dense = vectorized.toarray()

    # Prediction
    prediction = model.predict(vectorized_dense)[0]

    # Convert number -> label
    sentiment = label_encoder.inverse_transform([prediction])[0]

    # Probability
    probabilities = model.predict_proba(vectorized_dense)[0]

    return sentiment, probabilities
# Test cases
test_reviews = [
    "This phone is AMAZING!!! Best purchase ever 😊",
    "Terrible product, waste of money",
    "It's okay, nothing special",
    "Battery backup is unimaginable. Great phone for the price!",
    "Camera quality is very poor, disappointed"
]

print("\n" + "="*70)
print("TESTING PREDICTIONS ON RAW INPUT (Like Web App Will Do)")
print("="*70 + "\n")

for review in test_reviews:
    sentiment, probs = predict_sentiment(review)
    confidence = max(probs) * 100
    
    print(f"Review: {review}")
    print(f"Prediction: {sentiment} (Confidence: {confidence:.1f}%)")
    print(f"Probabilities: Neg={probs[0]:.2f}, Neu={probs[1]:.2f}, Pos={probs[2]:.2f}")
    print("-" * 70)

In [ ]:
# ===== 14. SAVE EVERYTHING FOR DEPLOYMENT =====

# Save the complete pipeline
# ===== SAVE MODEL =====

joblib.dump(model, 'xgboost_sentiment_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')

print("✓ XGBoost model saved!")
print("✓ TF-IDF vectorizer saved!")

# Save the preprocessing function separately (for documentation)
joblib.dump(preprocess_text, 'preprocess_function.pkl')
print("✓ Preprocessing function saved as: preprocess_function.pkl")

print("\n" + "="*70)
print("MODEL TRAINING COMPLETE!")
print("Files saved:")
print("  1. sentiment_pipeline.pkl  (TF-IDF + Logistic Regression)")
print("  2. preprocess_function.pkl (Text cleaning function)")
print("="*70)